# Using OpenMM simulations with NanoVer

This notebook demonstrates how to set up a molecular simulation with OpenMM and combine it with NanoVer to run as interactive molecular dynamics (iMD) simulation that can be viewed in the NanoVer iMD-XR client.

## OpenMM simulation setup

First we set up an OpenMM simulation (example adapted from [OpenMM documentation](https://docs.openmm.org/latest/userguide/application/02_running_sims.html)).

In [1]:
from openmm import unit, LangevinMiddleIntegrator
from openmm.app import ForceField, PME, HBonds, PDBFile, Simulation

pdb = PDBFile("../systems/17-ala.pdb")
forcefield = ForceField('amber19-all.xml', 'amber19/tip3pfb.xml')

system = forcefield.createSystem(
    pdb.topology,
    nonbondedMethod=PME,
    nonbondedCutoff=1*unit.nanometer,
    constraints=HBonds,
    removeCMMotion=False,
)

integrator = LangevinMiddleIntegrator(
    300*unit.kelvin,
    1/unit.picosecond,
    0.002*unit.picoseconds,
)

simulation = Simulation(pdb.topology, system, integrator)
simulation.context.setPositions(pdb.positions)

In [2]:
# NBVAL_SKIP
simulation.minimizeEnergy()

## NanoVer simulation and server setup

Next we wrap the OpenMM simulation in a NanoVer simulation (this provides iMD support and compatibility with NanoVer), and start a server providing access to the simulation over the network.

In [3]:
from nanover.openmm import OpenMMSimulation

omm_sim = OpenMMSimulation.from_simulation(simulation)

In [4]:
from nanover.app import OmniRunner

imd_runner = OmniRunner.with_basic_server(omm_sim, port=0, name="openmm example")
imd_runner.print_basic_info()
imd_runner.load(0)

Serving "openmm example" (ws://localhost:60079), discoverable on all interfaces on port 54545
Available simulations:
[0]: "Unnamed OpenMM Simulation"
Switched to [0]: "Unnamed OpenMM Simulation"
Switched to [0]: "Unnamed OpenMM Simulation"


## Interacting in virtual reality

Finally, we connect using the [NanoVer iMD-XR client](https://irl2.github.io/nanover-docs/installation.html#installing-the-imd-xr-client) to see and interact with the live OpenMM dynamics.

<video src="../figures/openmm-example.webm" controls>